# Task 1: Anti-Cheat Pre-Filter

- **Type:** Binary classification, target `is_cheater`
- **Metric:** F2 on the cheater class (**binary**). The Kaggle page says macro, but the LB moved +0.0177 when we
  switched to the binary-F2 threshold, exactly as predicted for binary (macro predicted −0.014). The decision
  threshold is tuned on OOF probabilities (≈ 0.16: recall-hungry).
- Final model (round 3, 2026-11-09): LightGBM on the 15 signal features, averaged over 5 seeds (section 4d).

Runs top-to-bottom on a CPU. Experiment history: `../experiments/task1/experiments.md`.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "common").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from common import metrics, paths, utils

TASK = 1
utils.seed_everything()
DATA = paths.task_data(TASK)
OUT = paths.outputs_dir(TASK)
print(DATA, DATA.exists())

/Users/finyakginshabu/Documents/CPE_3-1/CPE342/player-intelligence-system-cpe-342-ml/data/01_binary_classification True


## 1. Data card & load

In [2]:
print(paths.datacard(TASK))
train_raw = pd.read_csv(DATA / "train.csv")
test = pd.read_csv(DATA / "test.csv")
print("train", train_raw.shape, "test", test.shape)

TASK 1: ANTI-CHEAT DETECTION

TARGET VARIABLE: is_cheater (Binary)
Binary classification: 0=Legitimate Player, 1=Cheater

FEATURES (31 features):

1. kill_death_ratio (float)
   Ratio of kills to deaths

2. headshot_percentage (float)
   Percentage of kills that are headshots[1]

3. win_rate (float)
   Percentage of matches won

4. accuracy_score (float)
   Overall shooting accuracy score

5. kill_consistency (float)
   Variance in kill performance (higher = more consistent)

6. reaction_time_ms (float)
   Average reaction time in milliseconds

7. account_age_days (float)
   Days since account creation

8. level (float)
   Current player level

9. level_progression_speed (float)
   Rate of level advancement

10. friend_network_size (float)
    Number of in-game friends

11. reports_received (float)
    Number of player reports received

12. device_changes_count (float)
    Frequency of device/hardware changes

13. input_consistency_score (float)
    Consistency of mouse/keyboard input 

train (104000, 33) test (25889, 32)


In [3]:
# 13,184 train rows have no label (EDA insight 1): drop them.
train = train_raw[train_raw["is_cheater"].notna()].reset_index(drop=True)
y = train["is_cheater"].astype(int)
print(f"labeled rows: {len(train):,} (dropped {len(train_raw) - len(train):,}), positive rate: {y.mean():.4f}")

labeled rows: 90,816 (dropped 13,184), positive rate: 0.3493


## 2. EDA

The full EDA, with numbers, is in `../experiments/task1/experiments.md` (2026-11-08). Main points:
- `reports_received` acts as a step rule: 4 reports means 100% cheaters, 20+ means 0%.
- Higher skill means *less* likely to be a cheater.
- Missing values carry no target signal, but their per-column rates differ between train and test.
- About 16 of the 31 features look like noise.

In [4]:
FEATURES = [c for c in test.columns if c != "player_id"]
pd.DataFrame({"nan_train": train[FEATURES].isna().mean(), "nan_test": test[FEATURES].isna().mean()}).round(3).T

,kill_death_ratio,headshot_percentage,win_rate,accuracy_score,kill_consistency,reaction_time_ms,account_age_days,level,level_progression_speed,friend_network_size,...,communication_rate,team_play_score,buy_decision_score,map_knowledge,clutch_success_rate,first_blood_rate,survival_time_avg,damage_per_round,utility_usage_rate,crosshair_placement
nan_train,0.096,0.027,0.049,0.006,0.094,0.079,0.145,0.072,0.081,0.223,...,0.205,0.195,0.127,0.078,0.025,0.078,0.243,0.091,0.111,0.128
nan_test,0.053,0.141,0.137,0.023,0.098,0.053,0.006,0.015,0.056,0.047,...,0.110,0.081,0.111,0.006,0.135,0.101,0.131,0.128,0.141,0.028


## 3. Features

The baselines use the **31 raw features only**. The feature-engineering ideas (FE1–FE10 in
`../experiments/task1/experiments.md`) come in later experiments, so their gain can be measured against these baselines.

In [5]:
X = train[FEATURES]
X_test = test[FEATURES]
X.shape, X_test.shape

((90816, 31), (25889, 31))

## 4. Cross-validation

5-fold `StratifiedKFold(shuffle=True, random_state=42)`; all models use the same folds.
Test predictions are the mean of the 5 fold models.

Scores reported (binary F2 = the LB metric):
- `f2_binary`: at the threshold tuned on the full OOF set (slightly optimistic).
- `f2_binary_nested` ± `fold_std`: honest estimate. Each fold is scored at the threshold tuned on the *other* 4 folds' OOF.
- `f2_binary_at_0.5`: shows how much threshold tuning is worth.
- `f2_macro`: the macro version at the macro-optimal threshold, logged for reference only.

In [6]:
FOLDS = list(StratifiedKFold(n_splits=5, shuffle=True, random_state=utils.SEED).split(X, y))


def evaluate(name, oof):
    thr, f2b = metrics.best_fbeta_threshold(y, oof, beta=2, average="binary")
    pred = (oof >= thr).astype(int)
    fold_nested = []
    for tr, va in FOLDS:
        t, _ = metrics.best_fbeta_threshold(y.iloc[tr], oof[tr], beta=2, average="binary")
        fold_nested.append(metrics.f2(y.iloc[va], (oof[va] >= t).astype(int), average="binary"))
    return {
        "model": name,
        "auc": roc_auc_score(y, oof),
        "threshold": thr,
        "f2_binary": f2b,
        "f2_binary_nested": np.mean(fold_nested),
        "fold_std": np.std(fold_nested),
        "f2_binary_at_0.5": metrics.f2(y, (oof >= 0.5).astype(int), average="binary"),
        "f2_macro": metrics.best_fbeta_threshold(y, oof, beta=2, average="macro")[1],
        "pred_pos_rate": pred.mean(),
    }


results = []

### 4a. Baseline: Logistic Regression (in-class model)

Median imputation and scaling are fit inside each fold through the pipeline.

In [7]:
oof_lr = np.zeros(len(y))
test_lr = np.zeros(len(X_test))
for k, (tr, va) in enumerate(FOLDS):
    model = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                          LogisticRegression(max_iter=2000, random_state=utils.SEED))
    model.fit(X.iloc[tr], y.iloc[tr])
    oof_lr[va] = model.predict_proba(X.iloc[va])[:, 1]
    test_lr += model.predict_proba(X_test)[:, 1] / len(FOLDS)

utils.save_preds(TASK, "logreg", oof_lr, test_lr)
results.append(evaluate("logreg", oof_lr))
results[-1]

{'model': 'logreg',
 'auc': 0.764904643142399,
 'threshold': 0.1547779174901141,
 'f2_binary': 0.7643075523662733,
 'f2_binary_nested': np.float64(0.763753901394607),
 'fold_std': np.float64(0.0009869395414280752),
 'f2_binary_at_0.5': 0.48347813321589195,
 'f2_macro': 0.6833169235186416,
 'pred_pos_rate': np.float64(0.7816353946441156)}

### 4b. LightGBM

NaN is handled natively (no missing-value flags, per EDA). Early stopping on the validation fold uses logloss.

In [8]:
LGB_PARAMS = dict(
    objective="binary", learning_rate=0.03, num_leaves=31, min_child_samples=50,
    feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=1, lambda_l2=1.0,
    seed=utils.SEED, deterministic=True, force_row_wise=True, n_jobs=-1, verbose=-1,
)

oof_lgb = np.zeros(len(y))
test_lgb = np.zeros(len(X_test))
importance = pd.DataFrame(index=FEATURES)
for k, (tr, va) in enumerate(FOLDS):
    model = lgb.train(
        LGB_PARAMS,
        lgb.Dataset(X.iloc[tr], y.iloc[tr]),
        num_boost_round=5000,
        valid_sets=[lgb.Dataset(X.iloc[va], y.iloc[va])],
        callbacks=[lgb.early_stopping(200, verbose=False)],
    )
    oof_lgb[va] = model.predict(X.iloc[va], num_iteration=model.best_iteration)
    test_lgb += model.predict(X_test, num_iteration=model.best_iteration) / len(FOLDS)
    importance[f"fold{k}"] = model.feature_importance("gain")
    print(f"fold {k}: best_iter={model.best_iteration}, auc={roc_auc_score(y.iloc[va], oof_lgb[va]):.4f}")

utils.save_preds(TASK, "lgbm", oof_lgb, test_lgb)
results.append(evaluate("lgbm", oof_lgb))
results[-1]

fold 0: best_iter=789, auc=0.8915


fold 1: best_iter=823, auc=0.8913


fold 2: best_iter=872, auc=0.8920


fold 3: best_iter=640, auc=0.8872


fold 4: best_iter=575, auc=0.8938


{'model': 'lgbm',
 'auc': 0.8911634445457827,
 'threshold': 0.16398479608623748,
 'f2_binary': 0.8370170796764757,
 'f2_binary_nested': np.float64(0.8367873522212689),
 'fold_std': np.float64(0.0010242784969475213),
 'f2_binary_at_0.5': 0.63137991220183,
 'f2_macro': 0.7796754466775996,
 'pred_pos_rate': np.float64(0.650237843551797)}

In [9]:
imp = importance.mean(axis=1)
(imp / imp.sum()).sort_values(ascending=False).round(4).to_frame("gain_share").head(20)

,gain_share
crosshair_placement,0.2067
reports_received,0.1969
account_age_days,0.1609
friend_network_size,0.1322
level,0.0650
game_sense_score,0.0269
headshot_percentage,0.0232
kill_death_ratio,0.0207
level_progression_speed,0.0154
accuracy_score,0.0119


### 4c. LightGBM, 15 signal features, 5 seeds (round 3, 2026-11-09)
Every single model plateaus at OOF binary F2 ≈ 0.837 (feature packs, CatBoost, QDA features: `../experiments/task1/experiments.md`), so the
last lever is variance. Dropping the 16 noise features (EDA insight 7) and averaging 5 seeds (bagging/feature sampling
differ) smooths the probabilities around the recall-hungry threshold: nested binary F2 0.8368 → 0.8379, every fold up,
and the tuned threshold becomes identical in all 5 folds (0.159). A CatBoost blend adds nothing (`../experiments/task1/src/experiments_r3.py`).

In [10]:
NOISE = ["sessions_per_day", "first_blood_rate", "survival_time_avg", "damage_per_round", "device_changes_count",
         "clutch_success_rate", "map_knowledge", "night_play_ratio", "avg_session_length_min", "utility_usage_rate",
         "team_play_score", "buy_decision_score", "communication_rate", "weapon_switch_speed",
         "movement_pattern_score", "input_consistency_score"]
SIGNAL = [c for c in FEATURES if c not in NOISE]
SEEDS = [42, 43, 44, 45, 46]
oof_ms, test_ms = np.zeros(len(y)), np.zeros(len(X_test))
for s in SEEDS:
    for tr, va in FOLDS:
        model = lgb.train({**LGB_PARAMS, "seed": s}, lgb.Dataset(X.iloc[tr][SIGNAL], y.iloc[tr]), num_boost_round=5000,
                          valid_sets=[lgb.Dataset(X.iloc[va][SIGNAL], y.iloc[va])],
                          callbacks=[lgb.early_stopping(200, verbose=False)])
        oof_ms[va] += model.predict(X.iloc[va][SIGNAL], num_iteration=model.best_iteration) / len(SEEDS)
        test_ms += model.predict(X_test[SIGNAL], num_iteration=model.best_iteration) / (len(SEEDS) * len(FOLDS))
utils.save_preds(TASK, "lgbm_sig_5seed", oof_ms, test_ms)
results.append(evaluate("lgbm_sig_5seed", oof_ms))
results[-1]

{'model': 'lgbm_sig_5seed',
 'auc': 0.8924266402747087,
 'threshold': 0.15891262852808208,
 'f2_binary': 0.837937137231908,
 'f2_binary_nested': np.float64(0.8378754672271864),
 'fold_std': np.float64(0.001373087635350366),
 'f2_binary_at_0.5': 0.6339940535183349,
 'f2_macro': 0.7802803593427432,
 'pred_pos_rate': np.float64(0.6496322233967583)}

### 4d. Comparison

In [11]:
res = pd.DataFrame(results).set_index("model")
res.to_csv(OUT / "cv_results.csv")
res.round(4)

,auc,threshold,f2_binary,f2_binary_nested,fold_std,f2_binary_at_0.5,f2_macro,pred_pos_rate
model,,,,,,,,
logreg,0.7649,0.1548,0.7643,0.7638,0.0010,0.4835,0.6833,0.7816
lgbm,0.8912,0.1640,0.8370,0.8368,0.0010,0.6314,0.7797,0.6502
lgbm_sig_5seed,0.8924,0.1589,0.8379,0.8379,0.0014,0.6340,0.7803,0.6496


## 5. Final model & predictions

Pick the model with the best **nested** binary F2 (the honest estimate). Use its full-OOF binary-F2 threshold on
the fold-averaged test probabilities.

In [12]:
best = res["f2_binary_nested"].idxmax()
thr = res.loc[best, "threshold"]
test_prob = {"logreg": test_lr, "lgbm": test_lgb, "lgbm_sig_5seed": test_ms}[best]
test_label = (test_prob >= thr).astype(int)
print(f"best={best} threshold={thr:.4f}")
print(f"predicted positive rate: OOF {res.loc[best, 'pred_pos_rate']:.4f} vs test {test_label.mean():.4f}")

best=lgbm_sig_5seed threshold=0.1589
predicted positive rate: OOF 0.6496 vs test 0.6172


## 6. Write `outputs/final_pred.csv`

Keyed by `player_id`; `make_submission.py` maps test.csv row order to `ANS#####`.

In [13]:
final = utils.write_final_pred(TASK, test["player_id"], test_label)
final.head()

,id,prediction
0,P122189,1
1,P000044,1
2,P050138,1
3,P023303,1
4,P128406,0
